In [ ]:
import subprocess
import sys
import os
import shutil

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('scipy>=1.11.0', 'opencv-python-headless>=4.8.0', 'numpy>=1.24.0', 'ultralytics', 'huggingface_hub')

subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

YOLO_FACE_PATH = '/kaggle/working/yolov8n-face.pt'

if not os.path.exists(YOLO_FACE_PATH):
    from huggingface_hub import hf_hub_download
    tmp = hf_hub_download(repo_id='arnabdhar/YOLOv8-Face-Detection', filename='model.pt')
    shutil.copy(tmp, YOLO_FACE_PATH)

In [2]:
%%writefile /kaggle/working/pipeline_worker.py
import subprocess
import sys
import os
import shutil
import cv2
import json
import queue
import threading
import numpy as np
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Optional, Tuple, Dict
from tqdm.auto import tqdm
from ultralytics import YOLO

@dataclass
class BBox:
    top: float
    bottom: float
    left: float
    right: float

    def to_dict(self):
        return asdict(self)

    def width(self):
        return max(0.0, self.right - self.left)

    def height(self):
        return max(0.0, self.bottom - self.top)

    def area(self):
        return self.width() * self.height()

    def iou(self, other: 'BBox') -> float:
        ix1 = max(self.left, other.left)
        iy1 = max(self.top, other.top)
        ix2 = min(self.right, other.right)
        iy2 = min(self.bottom, other.bottom)
        inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
        union = self.area() + other.area() - inter
        return inter / union if union > 0 else 0.0

    def with_padding(self, pad: float) -> 'BBox':
        pw = self.width() * pad
        ph = self.height() * pad
        left = max(0.0, self.left - pw)
        right = min(1.0, self.right + pw)
        top = max(0.0, self.top - ph)
        bottom = min(1.0, self.bottom + ph)
        return BBox(top=top, bottom=bottom, left=left, right=right)

@dataclass
class Segment:
    start_sec: float
    end_sec: float
    start_frame: int
    end_frame: int

    @property
    def duration(self):
        return self.end_sec - self.start_sec

@dataclass
class ClipResult:
    youtube_id: str
    start_sec: float
    end_sec: float
    bbox: BBox

    def to_dict(self):
        return {
            'youtube_id': self.youtube_id,
            'duration': {
                'start_sec': round(self.start_sec, 4),
                'end_sec': round(self.end_sec, 4),
            },
            'bbox': self.bbox.to_dict()
        }

class FaceTracker:
    def track(self, detections: List[Optional[BBox]]) -> List[Optional[BBox]]:
        active = []
        tracks = []
        next_id = 0

        class Track:
            def __init__(self, tid, fi, bbox):
                self.tid = tid
                self.start = fi
                self.last_frame = fi
                self.last_bbox = bbox
                self.frames = {fi: bbox}

            def update(self, fi, bbox):
                self.last_frame = fi
                self.last_bbox = bbox
                self.frames[fi] = bbox

        for fi, bbox in enumerate(detections):
            active = [t for t in active if fi - t.last_frame <= 5]
            if bbox is None:
                continue
            best_t = None
            best_iou = 0.25
            for t in active:
                iou = bbox.iou(t.last_bbox)
                if iou > best_iou:
                    best_iou = iou
                    best_t = t
            if best_t:
                best_t.update(fi, bbox)
            else:
                t = Track(next_id, fi, bbox)
                next_id += 1
                tracks.append(t)
                active.append(t)

        if not tracks:
            return [None] * len(detections)

        main = max(tracks, key=lambda t: len(t.frames))
        result = [None] * len(detections)
        for fi, bbox in main.frames.items():
            result[fi] = bbox

        n = len(result)
        i = 0
        while i < n:
            if result[i] is not None:
                i += 1
                continue
            j = i
            while j < n and result[j] is None:
                j += 1
            gap_len = j - i
            if gap_len <= 5 and i > 0 and j < n:
                a = result[i-1]
                b = result[j]
                for k in range(gap_len):
                    t_val = (k + 1) / (gap_len + 1)
                    interp_bbox = BBox(
                        top=a.top + (b.top - a.top) * t_val,
                        bottom=a.bottom + (b.bottom - a.bottom) * t_val,
                        left=a.left + (b.left - a.left) * t_val,
                        right=a.right + (b.right - a.right) * t_val
                    )
                    result[i+k] = interp_bbox
            i = j + 1
        return result

class Pipeline:
    def __init__(self, output_dir, gpu_id=0):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.gpu_id = gpu_id
        self.device = f'cuda:{gpu_id}'
        self.model = YOLO('/kaggle/working/yolov8n-face.pt')
        self.model.to(self.device)
        self.tracker = FaceTracker()
        
    def check_h264(self, video_path: str) -> bool:
        cmd = ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=codec_name', '-of', 'default=noprint_wrappers=1:nokey=1', video_path]
        try:
            codec = subprocess.check_output(cmd, stderr=subprocess.DEVNULL, text=True).strip()
            return codec == 'h264'
        except Exception:
            return False

    def get_dimensions(self, path: str):
        cmd = ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=width,height', '-of', 'csv=p=0:s=x', path]
        try:
            out = subprocess.check_output(cmd, stderr=subprocess.DEVNULL, text=True).strip()
            w, h = out.split('x')
            return int(w), int(h)
        except Exception:
            cap = cv2.VideoCapture(path)
            w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            cap.release()
            return w, h

    def detect(self, cap, total_frames, fps) -> List[Optional[BBox]]:
        q = queue.Queue(maxsize=3)
        YOLO_DIM = 640
        batch_size = 1024

        def reader():
            buf = []
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                h, w = frame.shape[:2]
                scale = YOLO_DIM / max(h, w)
                frame = cv2.resize(frame, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_LINEAR)
                buf.append(frame)
                if len(buf) == batch_size:
                    q.put(buf)
                    buf = []
            if buf:
                q.put(buf)
            q.put(None)

        threading.Thread(target=reader, daemon=True).start()
        detections = []
        pbar = tqdm(total=total_frames, desc=f'GPU {self.gpu_id}', position=self.gpu_id, leave=True)

        while True:
            batch = q.get()
            if batch is None:
                break
            results = self.model.predict(batch, verbose=False, conf=0.4, half=True, stream=True)
            for i, res in enumerate(results):
                h, w = batch[i].shape[:2]
                boxes = res.boxes
                if boxes is None or len(boxes) == 0:
                    detections.append(None)
                else:
                    xyxy = boxes.xyxy.cpu().numpy()
                    areas = (xyxy[:,2] - xyxy[:,0]) * (xyxy[:,3] - xyxy[:,1])
                    best = xyxy[areas.argmax()]
                    detections.append(BBox(float(best[1]/h), float(best[3]/h), float(best[0]/w), float(best[2]/w)))
            pbar.update(len(batch))
        pbar.close()
        return detections

    def segment(self, tracked: List[Optional[BBox]], fps: float) -> List[Segment]:
        raw = []
        in_span = False
        span_start = 0
        for fi, b in enumerate(tracked):
            if b is not None and not in_span:
                span_start = fi
                in_span = True
            elif b is None and in_span:
                raw.append((span_start, fi - 1))
                in_span = False
        if in_span:
            raw.append((span_start, len(tracked) - 1))

        merged = []
        max_gap_f = int(0.5 * fps)
        for s in raw:
            if merged and s[0] - merged[-1][1] <= max_gap_f:
                merged[-1] = (merged[-1][0], s[1])
            else:
                merged.append(list(s))

        segs = []
        for sf, ef in merged:
            win = tracked[sf:ef+1]
            vis = sum(1 for b in win if b is not None) / max(len(win), 1)
            if vis < 0.85:
                continue
            max_len_f = int(6.0 * fps)
            current_sf = sf
            while current_sf <= ef:
                current_ef = min(current_sf + max_len_f - 1, ef)
                dur = (current_ef - current_sf + 1) / fps
                if dur >= 2.0:
                    segs.append(Segment(current_sf/fps, current_ef/fps, current_sf, current_ef))
                current_sf = current_ef + 1
        return segs

    def smooth(self, seg: Segment, tracked: List[Optional[BBox]]) -> BBox:
        win = tracked[seg.start_frame: seg.end_frame + 1]
        if not win:
            return BBox(0.1, 0.9, 0.3, 0.7)
        
        n = len(win)
        arr = np.full((n, 4), np.nan, dtype=np.float32)
        for i, b in enumerate(win):
            if b is not None:
                arr[i] = [b.top, b.bottom, b.left, b.right]
        
        mask = np.isnan(arr[:, 0])
        idx = np.where(~mask, np.arange(n), 0)
        np.maximum.accumulate(idx, out=idx)
        arr = arr[idx]
        
        mask = np.isnan(arr[:, 0])
        if mask.any():
            idx = np.where(~mask, np.arange(n), n-1)
            idx = np.minimum.accumulate(idx[::-1])[::-1]
            arr = arr[idx]

        w = min(21, n)
        w = w if w % 2 else w - 1
        if w >= 3:
            x = np.linspace(-(w//2), w//2, w)
            k = np.exp(-0.5 * (x / (w/6.0))**2)
            k /= k.sum()
            out = np.empty_like(arr)
            for c in range(4):
                out[:,c] = np.convolve(arr[:,c], k, mode='same')
            arr = out

        lo, hi = max(0, int(n*.1)), min(n, int(n*.9))
        m = arr[lo:hi].mean(axis=0) if hi > lo else arr.mean(axis=0)
        return BBox(float(np.clip(m[0],0,1)), float(np.clip(m[1],0,1)), float(np.clip(m[2],0,1)), float(np.clip(m[3],0,1)))

    def process(self, video_path: str):
        if not self.check_h264(video_path):
            print(f'Skipping non-H264 video: {video_path}')
            return

        print(f'Processing {video_path} on GPU {self.gpu_id}')
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        detections = self.detect(cap, total, fps)
        cap.release()

        tracked = self.tracker.track(detections)
        segments = self.segment(tracked, fps)

        if not segments:
            print(f'No valid segments found in {video_path}')
            return

        W, H = self.get_dimensions(video_path)
        manifest = {}
        vid_stem = Path(video_path).stem

        for idx, seg in enumerate(segments, 1):
            bbox = self.smooth(seg, tracked)
            padded = bbox.with_padding(0.35)
            
            crop_x = int(padded.left * W)
            crop_y = int(padded.top * H)
            crop_w = max(2, int(padded.width() * W))
            crop_h = max(2, int(padded.height() * H))
            crop_w = crop_w if crop_w % 2 == 0 else crop_w - 1
            crop_h = crop_h if crop_h % 2 == 0 else crop_h - 1
            crop_x = max(0, min(crop_x, W - crop_w))
            crop_y = max(0, min(crop_y, H - crop_h))

            clip_id = f'{vid_stem}_clip_{idx:03d}'
            out_path = str(self.output_dir / f'{clip_id}.mp4')

            cmd = [
                'ffmpeg', '-y',
                '-ss', f'{seg.start_sec:.4f}',
                '-i', video_path,
                '-t', f'{seg.end_sec - seg.start_sec:.4f}',
                '-vf', f'crop={crop_w}:{crop_h}:{crop_x}:{crop_y}',
                '-c:v', 'libx264', '-crf', '22', '-preset', 'fast',
                '-c:a', 'aac', '-b:a', '128k',
                '-movflags', '+faststart',
                out_path
            ]
            subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            
            manifest[clip_id] = ClipResult(
                youtube_id=vid_stem,
                start_sec=seg.start_sec,
                end_sec=seg.end_sec,
                bbox=bbox
            ).to_dict()

        mpath = self.output_dir / 'manifest.json'
        existing = json.loads(mpath.read_text()) if mpath.exists() else {}
        existing.update(manifest)
        mpath.write_text(json.dumps(existing, indent=2, ensure_ascii=False))
        print(f'Completed {vid_stem} - {len(segments)} clips extracted')

def worker_process(video_path_str, gpu_queue):
    gpu_id = gpu_queue.get()
    try:
        video_path = Path(video_path_str)
        output_directory = f'/kaggle/working/clips/{video_path.stem}'
        pipeline = Pipeline(output_dir=output_directory, gpu_id=gpu_id)
        pipeline.process(str(video_path))
    finally:
        gpu_queue.put(gpu_id)

Writing /kaggle/working/pipeline_worker.py


In [ ]:
sys.path.append('/kaggle/working')
import concurrent.futures
import multiprocessing as mp
from pathlib import Path
import pipeline_worker

input_dirs = [
    "/kaggle/input/datasets/jousefna/egyptian-videos-raw-dataset",
    "/kaggle/input/notebooks/jousefna/process-video-codec-p1/h264_out",
    "/kaggle/input/notebooks/jousefna/process-video-codec-p2/h264_out",
    "/kaggle/input/notebooks/jousefna/process-video-codec-p3/h264_out",
    "/kaggle/input/notebooks/jousefna/process-video-codec-p4/h264_out"
]

all_videos = []
for d in input_dirs:
    all_videos.extend([p for p in Path(d).rglob('*') if p.suffix.lower() in {'.mp4', '.mkv', '.mov', '.avi', '.webm'}])

try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass

if __name__ == '__main__':
    with mp.Manager() as manager:
        gpu_queue = manager.Queue()
        gpu_queue.put(0)
        gpu_queue.put(1)

        with concurrent.futures.ProcessPoolExecutor(max_workers=2) as executor:
            futures = [executor.submit(pipeline_worker.worker_process, str(video), gpu_queue) for video in all_videos]
            concurrent.futures.wait(futures)

    print('Batch processing complete.')